## EU REED

#### 1) Import packages, background databases and set project

In [1]:
import bw2data as bd
import bw2calc as bc
import bw2io as bi
from pathlib import Path
import pandas as pd
import numpy as np
import seaborn as sns
import bw2analyzer as bwa
from reed.utils import Config
import bw_processing as bwp
from bw2io.strategies.generic import set_code_by_activity_hash


import matplotlib.pyplot as plt
import d3graph
import polyviz

from datetime import date

10:30:37+0200 [warning  ] Can't import `SimaProBlockCSVImporter` - please install `bw2io` with `pip install bw2io[multifunctional]` or install `multifunctional` and `bw_simapro_csv` manually.


In [2]:
bi.remote.install_project(project_key="ecoinvent-3.10-biosphere", overwrite_existing=True)

Restoring project backup archive - this could take a few minutes...
Restored project: ecoinvent-3.10-biosphere


'ecoinvent-3.10-biosphere'

In [3]:
config = Config()

In [4]:
bd.projects.set_current(config.PROJECT_NAME)

In [5]:
import import_db

#### 2) Import foreground database 

In [6]:
bonsai = bd.Database('BONSAI_V2.1.3')
bonsai_bio = bd.Database('BONSAI_V2.1.3 biosphere')
ecoinvent = bd.Database('ecoinvent-3.10-biosphere')

In [7]:
bd.databases

Databases dictionary with 4 object(s):
	BONSAI_V2.1.3
	BONSAI_V2.1.3 biosphere
	ecoinvent-3.10-biosphere
	reed

In [8]:
from bw2io.importers import ExcelImporter
from reed.utils import Config

importer  = ExcelImporter(config.REED_PATH/'model'/'REED_model_20260407.xlsx',)
importer.apply_strategies()
importer.match_database(fields=['name','unit','location'])
importer.match_database(db_name=bonsai.name,fields=['name','unit','location','activity type'])
importer.match_database(db_name=bd.config.biosphere,fields=['name','categories','unit'])

Extracted 6 worksheets in 0.85 seconds
Applying strategy: csv_restore_tuples
Applying strategy: csv_restore_booleans
Applying strategy: csv_numerize
Applying strategy: csv_drop_unknown
Applying strategy: csv_add_missing_exchanges_section
Applying strategy: normalize_units
Applying strategy: strip_biosphere_exc_locations
Applying strategy: set_code_by_activity_hash
Applying strategy: link_iterable_by_fields
Applying strategy: assign_only_product_as_production
Applying strategy: link_technosphere_by_activity_hash
Applying strategy: drop_falsey_uncertainty_fields_but_keep_zeros
Applying strategy: convert_uncertainty_types_to_integers
Applying strategy: convert_activity_parameters_to_list
Applied 14 strategies in 0.51 seconds
Applying strategy: link_iterable_by_fields
Applying strategy: link_iterable_by_fields
Applying strategy: link_iterable_by_fields


In [9]:
importer.statistics()

Graph statistics for `reed` importer:
77 graph nodes:
	None: 77
423 graph edges:
	technosphere: 344
	production: 77
	biosphere: 2
423 edges to the following databases:
	BONSAI_V2.1.3: 240
	reed: 181
	ecoinvent-3.10-biosphere: 2
0 unique unlinked edges (0 total):




(77, 423, 0, 0)

In [10]:
importer.write_database()

100%|██████████| 77/77 [00:00<00:00, 436.49it/s]


10:39:24+0200 [info     ] Vacuuming database            
Created database: reed


In [11]:
pd.DataFrame(importer.unlinked)

""


In [13]:
reed = bd.Database('reed')

In [14]:
data_reed = [activity.as_dict() for activity in reed]
df_reed = pd.DataFrame(data_reed)


In [15]:
from IPython.display import display
display(df_reed[['code', 'comment', 'location', 'unit', 'database']])

,code,comment,location,unit,database
0,402ca9af225c3a39030f068a0fd7ee07,This process represents flamecutting. The data...,DE,hour,reed
1,40f1d6ffdea9bb0ab793fa74c0897c02,None,GB,cubic meter,reed
2,da6c1abbf40f7ea771fa84c8d1aca9e9,"draft for manual welding process (MIG), missin...",GB,hour,reed
3,aaf91911068752e9ed2a6c4fd48326ea,to be used as proxy when machining process is ...,DE,kilogram,reed
4,9a004d0f93a2eb4675e3d3facaff510d,None,ES,ton kilometer,reed
...,...,...,...,...,...
72,8b3c0e5ddddcd1f502802a085213d786,proxy for drilling. There are multiple differe...,GB,kilogram,reed
73,c1893a1ff028fac62c5fd996db2753e0,proxy for turning. Water and other liquids for...,GB,kilogram,reed
74,4428f445499f433efd9573c4b43384dc,None,DE,unit,reed
75,2a2244b4efd30e69cfb176792c0c8d06,This process represents the production of oxyg...,DE,cubic meter,reed


In [16]:
bd.databases

Databases dictionary with 4 object(s):
	BONSAI_V2.1.3
	BONSAI_V2.1.3 biosphere
	ecoinvent-3.10-biosphere
	reed

#### 3) Importing sensordata
###### This part draws from defined functions in the script, sensordata_analysis.py. It imports the monitoring data and calculates the energy consumption per tool and connect the tools to the reed tools

In [17]:
from reed.sensordata_analysis import (
    run_sensordata_analysis,
    find_missing_op_type_tools,
    compute_event_totals,
    split_final_df_by_company_event,
    sum_by_reed_tool,
    split_final_df_by_company_event,
)

In [18]:
monitoring_data_path = config.REED_PATH/'Data'/"MonitoringData"/"Data"
tool_descriptions_path = config.REED_PATH/'Data'/"MonitoringData"

In [19]:
sensordata, event_summaries_df, ambiguous_company_tools, key_company_tools, key_reed_tools = run_sensordata_analysis(
    monitoring_data_path,
    tool_descriptions_path,
)
print(sensordata)

C:\Users\OskarLilleoere\repos_20LCA\reed\src\reed\sensordata_analysis.py:132: DtypeWarning: Columns (22) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(f)


                                                  energy_consumption  \
company event_id                     Tool_Number                       
goimek  OF10001_G_BQC_S8CF2G         0.0                    8.154403   
                                     1.0                   24.300319   
                                     2.0                   10.289347   
                                     3.0                   14.673875   
                                     4.0                    0.930889   
...                                                              ...   
wwr     OF10005_SETUPS2_G_TWB_7SZ421 2694.0                 2.915730   
                                     2695.0                 1.068689   
                                     2696.0                 2.709187   
                                     2722.0                 0.631658   
                                     3350.0                23.502143   

                                                  Power_Active 

Identify tools that have different operation type in the key_company_tools. These tool numbers are removed from the key to avoid an ambigious key. These tool numbers will be assigned to the generic machining process in the reed database

In [20]:
print(ambiguous_company_tools)

   company  Tool_Number                   op_type_candidates
0   goimek            2                 [AUXILIARY, MILLING]
1   goimek            6                 [AUXILIARY, MILLING]
2   goimek           30                 [AUXILIARY, MILLING]
3   goimek           31                   [CHAMFER, MILLING]
4   goimek           44                 [AUXILIARY, MILLING]
5   goimek           45                 [AUXILIARY, MILLING]
6  komatsu            3           [AUXILIARY, ROUGH MILLING]
7  komatsu           48  [AUXILIARY, MILLING, ROUGH MILLING]
8  komatsu           50        [ROUGH BORING, ROUGH MILLING]
9  komatsu           72      [FINISH MILLING, ROUGH MILLING]


In [21]:
#Identify missing op type tools, e.g. tools that are in the monitoring data but not in tool descriptions
missing_tool_descriptions = find_missing_op_type_tools(sensordata)

In [22]:
#Compute total time and energy per event per company
event_totals = compute_event_totals(sensordata, event_summaries_df)

#### 4) Switching in sensordata

this section is missing a part about how to handle the processes that are not in hours?

In [61]:
from reed.swapping_data import (
    find_activities_by_name,
    build_scenarios_from_power_active,
    build_consumer_input_scenarios_from_hours,
    describe_scenarios,
    lcia_for_activity_with_scenario,
    find_subactivity_by_name_fragment,
    build_consumer_input_scenarios_from_hours_for_activity,
    get_product_id_for_activity,
)

Select which event you want to analyse

In [25]:
company = "komatsu"
event_id = "OF10006_G_SLG_E8AJ1C"
product = "KOMATSU total"
demand = 1

Start by selected the company and event_id where you want to swap in the sensordata

In [26]:
#Split sensordata by company and event
event_dfs = split_final_df_by_company_event(sensordata)

#Compute energy by reed tool for a specific company and event
tool_energy_by_company_and_event = sum_by_reed_tool(
    event_dfs[(company, event_id)]
)
print(tool_energy_by_company_and_event)

                                                          energy_consumption  \
company event_id             reed tool                                         
komatsu OF10006_G_SLG_E8AJ1C generic machining, soraluce           79.911755   
                             milling, soraluce                     25.856973   

                                                              hours  \
company event_id             reed tool                                
komatsu OF10006_G_SLG_E8AJ1C generic machining, soraluce  23.718056   
                             milling, soraluce             6.878611   

                                                          Power_Active  
company event_id             reed tool                                  
komatsu OF10006_G_SLG_E8AJ1C generic machining, soraluce      3.369237  
                             milling, soraluce                3.759040  



Find all processes present in the sensordata in the reed-database

In [27]:
tools = tool_energy_by_company_and_event.index.get_level_values("reed tool").unique()
found = {t: find_activities_by_name(reed, t, exact=False) for t in tools}
found

{'generic machining, soraluce': ['generic machining, soraluce' (kilogram, GB, None),
  'generic machining, soraluce' (kilogram, DE, None),
  'generic machining, soraluce' (hour, GB, None),
  'generic machining, soraluce' (hour, ES, None),
  'generic machining, soraluce' (kilogram, ES, None),
  'generic machining, soraluce' (hour, DE, None)],
 'milling, soraluce': ['milling, soraluce' (hour, DE, None),
  'milling, soraluce' (kilogram, GB, None),
  'milling, soraluce' (kilogram, DE, None),
  'milling, soraluce' (hour, ES, None),
  'milling, soraluce' (kilogram, ES, None),
  'milling, soraluce' (hour, GB, None)]}

Find subactivities called A3 for the product under study

In [39]:
product_activity = reed.get(name=product)

# Find the A3 activity inside KOMATSU total
a3_matches = find_subactivity_by_name_fragment(
    parent_activity=product_activity,
    name_fragment="A3",
)

print(a3_matches)
a3_activity = a3_matches[0]

['KOMATSU A3: machining' (unit, GB, None)]


In [43]:
soraluce_matches = find_subactivity_by_name_fragment(
    parent_activity=a3_activity,
    name_fragment="soraluce machining",
)

print(soraluce_matches)

soraluce_activity = soraluce_matches[0]

['Soraluce machining, Komatsu piece' (unit, GB, None)]


In [42]:
#Identifying the electricity activity in bonsai to be able to compute the original electricity input amount for the activities found above
electricity = bonsai.get(name='production of electricity (market for)', location='GB', unit='TJ')

Create scenario where alle electricity inputs from the sensordata to the processes identified in found.

In [38]:
scenarios = build_scenarios_from_power_active(tool_energy_by_company_and_event, found, electricity)
scenarios


original  komatsu_OF10006_G_SLG_E8AJ1C
product            activity                                                  
304606049204039691 307798009263812608  0.000015                      0.000012
                   307798009419001856  0.000000                      0.000014

The input of products in found to consuming activities are found and used to build scenarios.

In [44]:
consumer_scenarios = build_consumer_input_scenarios_from_hours_for_activity(
    tool_energy_by_company_and_event,
    found,
    consumer_activity=soraluce_activity,
)
consumer_scenarios

,,original,komatsu_OF10006_G_SLG_E8AJ1C
product,activity,,
307798009263812608,307798009582579713,6.77439,23.718056


In [45]:
full_scenario = pd.concat([scenarios, consumer_scenarios])

In [46]:
full_scenario_readable = describe_scenarios(full_scenario)
full_scenario_readable

,product_name,product_unit,activity_name,activity_unit,original,komatsu_OF10006_G_SLG_E8AJ1C
0,production of electricity (market for),TJ,"generic machining, soraluce",hour,0.000015,0.000012
1,production of electricity (market for),TJ,"milling, soraluce",hour,0.000000,0.000014
2,"generic machining, soraluce",hour,"Soraluce machining, Komatsu piece",unit,6.774390,23.718056


Calculate a LCA score for a specific activity and scenario

In [47]:
score, lca = lcia_for_activity_with_scenario(
    activity=product_activity,
    method=config.lcia_gwp100,
    scenario_df=full_scenario,
    scenario_col=None,     # "original" or None (for defined scenario)
    amount=demand
)
score


11586.725739433698

Baseline LCA score for same product

In [ ]:
product_reed = product_activity
lca = product_reed.lca(method=config.lcia_gwp100)
lca.score

11237.99292720747

#### 5) Align data to Digital Product Passport (DPP)

In [74]:
def lcia_score_for_named_input(
    parent_activity,
    name_fragment,
    method,
    scenario_df=None,
    scenario_col=None,
    amount=1.0,
):
    """
    Calculate LCIA score for direct inputs of parent_activity whose names contain name_fragment.
    Applies scenario_df if provided.
    """
    matches = [
        exc for exc in parent_activity.technosphere()
        if name_fragment.lower() in exc.input.get("name", "").lower()
    ]

    if not matches:
        raise ValueError(
            f"No input containing '{name_fragment}' found in {parent_activity.get('name')}"
        )

    total_score = 0

    for exc in matches:
        input_activity = exc.input
        input_amount = float(exc["amount"]) * amount

        if scenario_df is None:
            fu, data_objs, _ = bd.prepare_lca_inputs(
                {input_activity: input_amount},
                method=method,
            )

            lca = bc.LCA(fu, method=method, data_objs=data_objs)
            lca.lci()
            lca.lcia()

            total_score += lca.score

        else:
            score, _ = lcia_for_activity_with_scenario(
                activity=input_activity,
                method=method,
                scenario_df=scenario_df,
                scenario_col=scenario_col,
                amount=input_amount,
            )

            total_score += score

    return total_score

In [75]:
score_A1 = lcia_score_for_named_input(
    parent_activity=product_activity,
    name_fragment="A1",
    method=config.lcia_gwp100,
    scenario_df=full_scenario,
    amount=demand,
)

score_A2 = lcia_score_for_named_input(
    parent_activity=product_activity,
    name_fragment="A2",
    method=config.lcia_gwp100,
    scenario_df=full_scenario,
    amount=demand,
)

score_A3 = lcia_score_for_named_input(
    parent_activity=product_activity,
    name_fragment="A3",
    method=config.lcia_gwp100,
    scenario_df=full_scenario,
    amount=demand,
)

In [80]:
score_total = score_A1 + score_A2 + score_A3
score_total

11871.790188809908

In [86]:
activity = reed.get(name=product)
location = activity.get("location")
unit = activity.get("unit")

def make_pcf_entry(score, life_cycle_phase):
    return {
        "PcfCalculationMethods": [
            {
                "PcfCalculationMethod": "ISO 14044"
            }
        ],
        "PcfCO2eq": round(score, 1),
        "ReferenceValueForCalculation": unit,
        "QuantityOfMeasureForCalculation": demand,
        "LifeCyclePhases": [
            {
                "LifeCyclePhase": life_cycle_phase
            }
        ],
        "ExplanatoryStatement": (
            "Consequential approach modelling the climate impact of demanding "
            "the specific product."
        ),
        "GoodsHandoverAddress": {
            "Country": location
        },
        "PublicationDate": date.today().isoformat(),
        # "ExpirationDate": "2027-03-06"
    }

dpp_output = {
    "DPP": [
        {
            "company": company,
            "event_id": event_id,
            "product": product,
            "carbonFootprint": {
                "ProductCarbonFootprints": [
                    make_pcf_entry(score_A1, "A1"),
                    make_pcf_entry(score_A2, "A2"),
                    make_pcf_entry(score_A3, "A3"),
                    make_pcf_entry(score_A1 + score_A2 + score_A3, "A1-A3")  # optional total
                ]
            }
        }
    ]
}

dpp_output


{'DPP': [{'company': 'komatsu',
   'event_id': 'OF10006_G_SLG_E8AJ1C',
   'product': 'KOMATSU total',
   'carbonFootprint': {'ProductCarbonFootprints': [{'PcfCalculationMethods': [{'PcfCalculationMethod': 'ISO 14044'}],
      'PcfCO2eq': 10489.8,
      'ReferenceValueForCalculation': 'unit',
      'QuantityOfMeasureForCalculation': 1,
      'LifeCyclePhases': [{'LifeCyclePhase': 'A1'}],
      'ExplanatoryStatement': 'Consequential approach modelling the climate impact of demanding the specific product.',
      'GoodsHandoverAddress': {'Country': 'GB'},
      'PublicationDate': '2026-04-29'},
     {'PcfCalculationMethods': [{'PcfCalculationMethod': 'ISO 14044'}],
      'PcfCO2eq': 110.1,
      'ReferenceValueForCalculation': 'unit',
      'QuantityOfMeasureForCalculation': 1,
      'LifeCyclePhases': [{'LifeCyclePhase': 'A2'}],
      'ExplanatoryStatement': 'Consequential approach modelling the climate impact of demanding the specific product.',
      'GoodsHandoverAddress': {'Country':

#### 6) Select processes to be analysed

In [ ]:
reed.search('casting')

In [ ]:
bonsai.search('electricity')

In [ ]:
bonsai_bio.search('carbon')

In [ ]:
#check_activity_type = print(lpg["activity type"])

In [ ]:
machining_processes = {
    "flamecutting, lpg": reed.get(name='flamecutting, lpg', location="DE", unit='meter'),
    "flamecutting, natural gas": reed.get(name='flamecutting, natural gas', location="DE", unit='meter'),
    "flamecutting, lpg": reed.get(name='flamecutting, lpg', location="DE", unit='hour'),
    "flamecutting, natural gas": reed.get(name='flamecutting, natural gas', location="DE", unit='hour'),
    "bending": reed.get(name='bending', location="DE", unit='hour'),
    "sawing": reed.get(name='sawing', location="DE", unit='hour'),
    "welding, manual": reed.get(name='welding, manual', location="GB", unit='hour'),
    "welding, robotic": reed.get(name='welding, robotic', location="GB", unit='hour'),
    "turning": reed.get(name='turning', location="GB", unit='kilogram'),
    "painting": reed.get(name='painting', location="DE", unit='m2'),
    "Casting": reed.get(name='Casting', location="DE", unit='ton'),
    "sandblasting": reed.get(name='sandblasting', location="DE", unit='m2'),
}

In [ ]:
soraluce_machining = {
    "generic machining, soraluce": reed.get(name='generic machining, soraluce', location="GB", unit='kilogram'),
    "generic machining, soraluce": reed.get(name='generic machining, soraluce', location="GB", unit='hour'),
    "milling, soraluce": reed.get(name='milling, soraluce', location="GB", unit='kilogram'),
    "milling, soraluce": reed.get(name='milling, soraluce', location="GB", unit='hour'),
    "drilling, soraluce": reed.get(name='drilling, soraluce', location="GB", unit='kilogram'),
    "drilling, soraluce": reed.get(name='drilling, soraluce', location="GB", unit='hour'),
}

In [ ]:
use_case_machines = {
    "use_case_komatsu": reed.get(name='Komatsu piece', location="GB", unit='unit'),
    "use_case_wwr": reed.get(name='WWR piece', location="DE", unit='unit'),
    "use_case_goimek": reed.get(name='Goimek piece', location="ES", unit='unit'),
}


In [ ]:
soraluce_machines = {
"soraluce_komatsu": reed.get(name='soraluce machine, Komatsu', location="GB", unit='unit'),
"soraluce_wwr": reed.get(name='soraluce machine, WWR', location="DE", unit='unit'),  
"soraluce_goimek": reed.get(name='soraluce machine, Goimek', location="ES", unit='unit'),
"Soraluce, average": reed.get(name='Soraluce, average', location="GB", unit='unit'),
}

In [ ]:
# export the LCI to Microsoft Excel in an overview format
#bi.export.excel.write_lci_excel(reed.name, objs=[steel], dirpath=Path.cwd())

#### 4) Defining LCIA functions

In [ ]:
def lca_score(process_dict, method):
    results = []
    for name, process in process_dict.items():
        lca = process.lca(method)
        results.append({
            "Process": name,
            "Location": process.get('location'),
            "Functional Unit": process.get('unit'),
            "Climate Impact (kg CO2eq)": lca.score
        })

    return pd.DataFrame(results)

In [77]:
def run_contribution_analysis(demand_object,
                              lcia_method,
                              max_level=1,
                              cutoff=0.01):
    return bwa.utils.recursive_calculation_to_object(
        demand_object,
        lcia_method=lcia_method,
        max_level=max_level,
        cutoff=cutoff,
        as_dataframe=True,
    )

In [ ]:
def plot_contribution(df,
                      title='Contribution Analysis',
                      xlabel='CO2 eq. emissions (kg)',
                      drop_parent=True):
    df_plot = df.copy()

    # Drop the parent (the demand itself)
    if drop_parent:
        # Heuristic: parent is usually the one with maximum score
        parent_score = df_plot['score'].max()
        df_plot = df_plot[df_plot['score'] < parent_score]

    # Sort descending by contribution
    df_plot = df_plot.sort_values(by='score', ascending=False)

    f, ax = plt.subplots(figsize=(5, 2))
    sns.barplot(y='name', x='score', data=df_plot, ax=ax)

    ax.set(
        title=title,
        xlabel=xlabel,
        ylabel=''
    )
    plt.tight_layout()
    plt.show()

#### 5) Calculate and present results

In [ ]:
df_results = lca_score(machining_processes, config.lcia_gwp100)
display(df_results)


In [ ]:
df_plot = df_results.set_index("Process")

ax = df_plot.plot(kind="bar", figsize=(10, 5))

ax.set_title("LCA Results (GWP100)")
ax.set_xlabel("Process")
ax.set_ylabel("GWP100 score")

plt.tight_layout()
plt.show()

In [ ]:
flamecutting = reed.get(name = 'flamecutting, lpg', location = "DE", unit = 'hour')

In [ ]:
# Another option is to use the recursive calculation to an object, which returns a DataFrame
flamecutting_ca = bwa.utils.recursive_calculation_to_object(flamecutting,
                                          lcia_method=config.lcia_gwp100,
                                          max_level=3,
                                          cutoff=0.001,
                                          as_dataframe=True,
                                          )
flamecutting_ca

In [ ]:
df_results = lca_score(soraluce_machining, config.lcia_gwp100)
display(df_results)


In [ ]:
df_plot = df_results.set_index("Process")

ax = df_plot.plot(kind="bar", figsize=(10, 5))

ax.set_title("LCA Results (GWP100)")
ax.set_xlabel("Process")
ax.set_ylabel("GWP100 score")

plt.tight_layout()
plt.show()

In [ ]:
df_results = lca_score(use_case_machines, config.lcia_gwp100)
display(df_results)

In [ ]:
df_plot = df_results.set_index("Process")

ax = df_plot.plot(kind="bar", figsize=(10, 5))

ax.set_title("LCA Results (GWP100)")
ax.set_xlabel("Process")
ax.set_ylabel("GWP100 score")

plt.tight_layout()
plt.show()

#### 6) Run contribution analysis

In [ ]:
processes_results_ca = {}

for key, demand in soraluce_machining.items():
    print(f"Running contribution analysis: {key}")
    
    df = run_contribution_analysis(
        demand_object=demand,
        lcia_method=config.lcia_gwp100,
        max_level=1,
        cutoff=0.01
    )
    processes_results_ca[key] = df
    
    plot_contribution(
        df,
        title=f"Contribution Analysis for {key.replace('_', ' ').title()}"
    )

In [ ]:
use_case_results_ca = {}

for key, demand in use_case_machines.items():
    print(f"Running contribution analysis: {key}")
    
    df = run_contribution_analysis(
        demand_object=demand,
        lcia_method=config.lcia_gwp100,
        max_level=1,
        cutoff=0.01
    )
    use_case_results_ca[key] = df
    
    plot_contribution(
        df,
        title=f"Contribution Analysis for {key.replace('_', ' ').title()}"
    )